# Agregação dos Dados 🎒 🎲

<pre>Algumas organizações oferecem bibliotecas para facilitar a criação de consultas, integrando-as ao nosso código.</pre>

<pre>A presente análise será realizada com base nos dados dos países pertencentes à <a href="https://databank.worldbank.org/source/world-development-indicators">base de dados do Banco Mundial</a>, que possuem a informação de PIB <i>per capita</i> (current US$) disponível para o ano de 2019.</pre>

## Explorando os dados econômicos do Banco Mundial

<pre>O Banco Mundial oferece biblioteca em Python para consultar seus bancos de dados.</pre>

👉 mais detalhes, consulte o pacote <a href='https://pypi.org/project/wbgapi/'>wbgapi</a> e sua <a href='https://blogs.worldbank.org/opendata/introducing-wbgapi-new-python-package-accessing-world-bank-data'>documentação</a>.

<pre>Vamos instalar e importar as bibliotecas que serão utilizadas neste caderno.</pre>

In [1]:
!pip install wbgapi --quiet
!pip install pandas --quiet

<pre>Execute as 2 células abaixo para montar o DataFrame <b>df</b>, com dados econômicos dos países.</pre>

In [2]:
import wbgapi as wb
import pandas as pd

In [3]:
economy_info = wb.economy.info()

countries = [pais for pais in economy_info.items if not pais.get('aggregate')]

countries_dict = {"country_code": [country.get("id") for country in countries],
                  "country_name": [country.get("value") for country in countries]}

countries_df = pd.DataFrame(countries_dict).set_index("country_code")

indicators_df = wb.data.DataFrame(series = ['NY.GDP.PCAP.CD', 'NE.GDI.FTOT.CD', 'SP.POP.TOTL', 
                                            'SL.TLF.ADVN.ZS',  'AG.LND.ARBL.HA.PC', 'NV.IND.TOTL.ZS', 
                                            'SE.XPD.TOTL.GD.ZS'],
                                  economy = countries_df.index,
                                  time = 2019)

df = countries_df.join(indicators_df)

df = df.rename(columns={"NY.GDP.PCAP.CD": "GDP_PC", "SL.TLF.ADVN.ZS" : "forca_trab_educ",
                        "AG.LND.ARBL.HA.PC": "arable_land", "NV.IND.TOTL.ZS": "industria_PERCPIB",
                        "SE.XPD.TOTL.GD.ZS": "gasto_educ_PERCPIB", "NE.GDI.FTOT.CD": "FBKF", 
                        "SP.POP.TOTL": "populacao"})

df.head()

,country_name,arable_land,FBKF,industria_PERCPIB,GDP_PC,gasto_educ_PERCPIB,forca_trab_educ,populacao
country_code,,,,,,,,
ABW,Aruba,0.018315,NaN,11.372037,31096.205074,4.435043,NaN,109203.0
AFG,Afghanistan,0.205726,NaN,14.058112,496.602504,NaN,NaN,37856121.0
AGO,Angola,0.165649,1.255763e+10,49.760922,2189.855714,2.073064,91.473,32375632.0
ALB,Albania,0.213721,3.834691e+09,22.965531,5460.428237,3.916240,75.465,2854191.0
AND,Andorra,0.010345,NaN,11.699818,41257.804585,3.150610,NaN,76474.0


<pre>Encontre os 5 países com as maiores rendas per capita (coluna/variável <b>GDP_PC</b>) do DataFrame <b>df</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df.sort_values(by='GDP_PC', ascending=False).head(5)
```

<br/>

</details>

<pre>Agora, encontre os 5 países com as menores rendas per capita (coluna/variável <b>GDP_PC</b>) do DataFrame <b>df</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df.sort_values(by='GDP_PC',ascending=False).tail(5)
```

<br/>

</details>

<pre>Vamos dividir os países em 5 quintis, de acordo com seu nível de riqueza. Lembrando que um quintil é qualquer um dos valores de uma variável que divide o seu conjunto ordenado em cinco partes iguais, de modo que:
     - 0% a 20% dos valores ordenados de <b>GDP_PC</b> definem o grupo <b>pobres</b>;
     - 20% a 40% dos valores ordenados de <b>GDP_PC</b> definem o grupo <b>medio pobres</b>;
     - 40% a 60% dos valores ordenados de <b>GDP_PC</b> definem o grupo <b>medio</b>;
     - 60% a 80% dos valores ordenados de <b>GDP_PC</b> definem o grupo <b>medio ricos</b>;
     - 80% a 100% dos valores ordenados de <b>GDP_PC</b> definem o grupo <b>ricos</b>.</pre>

<pre>Portanto, <b>quintil</b> deve ser o nome da nova variável/coluna do DataFrame <b>df</b> que contém, como rótulos: <b>pobres</b>, <b>medio pobres</b>, <b>medio</b>, <b>medio ricos</b>, ou <b>ricos</b> para cada uma dos registro/linhas.

<details>
  <summary>Resposta</summary>

<br/>

```python
df['quintil'] = pd.qcut(df['GDP_PC'], 
                        q = 5, 
                        labels = ["pobres", "medio pobres", "medio", "medio ricos", "ricos"])
```

<br/>

</details>

👉 dica: para calcular os quintis automaticamente, use o método `qcut` do pandas 🐼, passando 3 parâmetros: a `Series` com a coluna/variável que se quer ordenar, o número de grupos (`5`), bem como os rótulos de cada grupo, em ordem.

<pre>O Brasil foi enquadrado em qual grupo?</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df.loc['BRA', 'quintil']
```

<br/>

</details>

### Aspectos de GroupBy 🎒 

<pre>Por fim, analise as médias de cada variável/coluna, que contém apenas dados numéricos, para cada um dos grupos definidos.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df.groupby('quintil').mean(numeric_only=True)
```

<br/>

</details>

👉 dica: use a coluna/variável `quintil` como parâmetro do `groupby` do DataFrame `df`, bem como o método `mean` com parâmetro `numeric_only` no resultado do `groupby`.

<pre>Quais conclusões você pode tirar dos dados?</pre>